# 01 — PTB-XL Data Exploration

Look at what we're actually working with before training anything:
site/domain distribution, class balance, and a first look at raw
waveforms — all in plain language, matching Section 6.1 of the proposal.

Assumes `00_colab_setup.ipynb` has already been run in this runtime.

In [ ]:
%cd /content/mm-dg-healthcare
import pandas as pd
import matplotlib.pyplot as plt
from src.data.ptbxl_dataset import load_ptbxl_metadata, DIAGNOSTIC_SUPERCLASSES

metadata = load_ptbxl_metadata("data/raw/ptbxl")
metadata.head()

## Domain-shift axis: recording site

This is the column we'll split on to simulate "train at one hospital,
test at another," the same idea as the reference paper's Center A /
Center B experiments.

In [ ]:
site_counts = metadata["site"].value_counts().sort_index()
site_counts.plot(kind="bar", title="Records per PTB-XL site (our domain-shift axis)")
plt.xlabel("Site ID")
plt.ylabel("Number of records")
plt.show()

print(site_counts)

## Class balance per site

Worth checking whether the class balance itself shifts across sites —
if one site has almost no examples of a class, that's a harder
domain-shift pair to learn from and a natural first "hard" experiment.

In [ ]:
pd.crosstab(metadata["site"], metadata["superclass"])

In [ ]:
ct = pd.crosstab(metadata["site"], metadata["superclass"], normalize="index")
ct.plot(kind="bar", stacked=True, figsize=(8, 5), title="Class proportions per site")
plt.ylabel("Proportion")
plt.legend(title="Superclass", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## A first look at a raw waveform

Sanity-check that a record actually loads and looks like an ECG before
we trust anything downstream.

In [ ]:
import wfdb
import numpy as np

row = metadata.iloc[0]
record_path = f"data/raw/ptbxl/{row['filename_lr']}"
signal, meta = wfdb.rdsamp(record_path)
print(f"Record: ecg_id={row['ecg_id']}, superclass={row['superclass']}, site={row['site']}")
print(f"Signal shape: {signal.shape} (samples, leads)")
print(f"Lead names: {meta['sig_name']}")

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, lead_idx, lead_name in zip(axes, [0, 1, 6], [meta['sig_name'][0], meta['sig_name'][1], meta['sig_name'][6]]):
    ax.plot(signal[:, lead_idx])
    ax.set_ylabel(lead_name)
axes[-1].set_xlabel("Sample (100 Hz)")
fig.suptitle(f"ECG {row['ecg_id']} — {row['superclass']}")
plt.tight_layout()
plt.show()

## Independent text modality preview (Phase 3 groundwork)

Even though fusion isn't built yet, we can already look at the
independent text each record maps to — this is the "second modality"
Phase 3 will use, and it's worth eyeballing now to judge how informative
(or how repetitive) it looks before investing in a text encoder.

In [ ]:
from src.data.ptbxl_text import PTBXLTextLookup

text_lookup = PTBXLTextLookup("data/raw/ptbxl")
for _, row in metadata.sample(5, random_state=42).iterrows():
    text = text_lookup.get_text(row["scp_codes"])
    print(f"[{row['superclass']}] {text}")

Continue to `02_baseline_signal_only.ipynb` to train the Phase 1 baseline.